In [1]:
import logging
import os
import sys
import time
import importlib
from collections import defaultdict
import string
from pathlib import Path

import circuitsvis as cv
import einops
import numpy as np
import torch as t
from IPython.display import display
from jaxtyping import Float
from nnsight import CONFIG, LanguageModel
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from torch import Tensor
import os

from dotenv import load_dotenv

def find_arena_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ARENA_3.0").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find an ARENA_3.0 folder above {start}")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part32_function_vectors_and_model_steering"

root_dir = find_arena_root(Path.cwd())
exercises_dir = root_dir / "ARENA_3.0" / chapter / "exercises"
section_dir = exercises_dir / section

if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))
    
os.chdir(section_dir)

load_dotenv()
# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)
logging.disable(sys.maxsize)


MAIN = __name__ == "__main__"

section_module = importlib.import_module(
    "part32_function_vectors_and_model_steering"
)
sys.modules["part42_function_vectors_and_model_steering"] = section_module


import part32_function_vectors_and_model_steering.solutions as solutions
import part32_function_vectors_and_model_steering.tests as tests
from plotly_utils import imshow

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

/Users/shubmeister/miniconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
model = LanguageModel("EleutherAI/gpt-j-6b", device_map="auto", dtype=t.bfloat16)
tokenizer = model.tokenizer

N_HEADS = model.config.n_head
N_LAYERS = model.config.n_layer
D_MODEL = model.config.n_embd
D_HEAD = D_MODEL // N_HEADS

print(f"Number of heads: {N_HEADS}")
print(f"Number of layers: {N_LAYERS}")
print(f"Model dimension: {D_MODEL}")
print(f"Head dimension: {D_HEAD}\n")

print("Entire config: ", model.config)

Number of heads: 16
Number of layers: 28
Model dimension: 4096
Head dimension: 256

Entire config:  GPTJConfig {
  "activation_function": "gelu_new",
  "architectures": [
    "GPTJForCausalLM"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "dtype": "bfloat16",
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gptj",
  "n_embd": 4096,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 28,
  "n_positions": 2048,
  "pad_token_id": null,
  "resid_pdrop": 0.0,
  "rotary": true,
  "rotary_dim": 64,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50,
      "temperature": 1.0
    }
  },
  "tie_word_embeddings": false,
  "tokenizer_class": "GPT2Tokenizer"

In [15]:
# Calling tokenizer returns a dictionary, containing input ids & other data.
# If returned as a tensor, then by default it will have a batch dimension.
print(tokenizer("This must be Thursday", return_tensors="pt"))

# Decoding a list of integers, into a concatenated string.
print(tokenizer.decode([40, 1239, 714, 651, 262, 8181, 286, 48971, 12545, 13]))

# Using batch decode, on both 1D and 2D input.
print(tokenizer.batch_decode([4711, 2456, 481, 307, 6626, 510]))
print(tokenizer.batch_decode([[1212, 6827, 481, 307, 1978], [2396, 481, 428, 530]]))

# Split sentence into tokens (note we see the special Ġ character in place of prepended spaces).
print(tokenizer.tokenize("This sentence will be tokenized"))

{'input_ids': tensor([[1212, 1276,  307, 3635]]), 'attention_mask': tensor([[1, 1, 1, 1]])}
I never could get the hang of Thursdays.
['These words will be split up']
['This sentence will be together', 'So will this one']
['This', 'Ġsentence', 'Ġwill', 'Ġbe', 'Ġtoken', 'ized']


In [16]:
REMOTE = True
if REMOTE:
    load_dotenv(override=True)
    CONFIG.set_default_api_key(os.getenv("KEY"))

prompt = "The Eiffel Tower is in the city of"

with model.trace(prompt, remote=REMOTE):
    # Save the model's hidden states
    hidden_states = model.transformer.h[-1].output[0].save()

    # Save the model's logit output
    logits = model.lm_head.output[0, -1].save()

# Get the model's logit output, and its next token prediction
print(f"logits.shape = {logits.shape} = (vocab_size,)")
print("Predicted token ID =", predicted_token_id := logits.argmax().item())
print(f"Predicted token = {tokenizer.decode(predicted_token_id)!r}")

# Print the shape of the model's residual stream
print(f"\nresid.shape = {hidden_states.shape} = (batch_size, seq_len, d_model)")

⬇ Downloading: 100%|██████████| 143k/143k [00:00<00:00]

logits.shape = torch.Size([50400]) = (vocab_size,)
Predicted token ID = 6342
Predicted token = ' Paris'

resid.shape = torch.Size([1, 10, 4096]) = (batch_size, seq_len, d_model)


In [17]:
with model.trace(prompt, remote=REMOTE):
    attn_patterns = model.transformer.h[0].attn.attn_dropout.input.save()
    
str_tokens = model.tokenizer.tokenize(prompt)
str_tokens = [s.replace('Ġ', ' ') for s in str_tokens]

# Attention patterns (squeeze out the batch dimension)
attn_patterns_value = attn_patterns.squeeze(0)

print("Layer 0 Head Attention Patterns:")
display(cv.attention.attention_patterns(
    tokens=str_tokens,
    attention=attn_patterns_value,
))

⬇ Downloading: 100%|██████████| 2.17k/2.17k [00:00<00:00]


Layer 0 Head Attention Patterns:


In [18]:
def generate_antonym_dataset(N: int):
    """
    Generates 100 pairs of antonyms, in the form of a list of 2-tuples.
    """
    assert os.environ.get("OPENAI_API_KEY", None) is not None, "Please set your API key before running this function!"

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": f"Generate {N} pairs of antonyms in the form of a list of 2-tuples. For example, [['old', 'young'], ['top', bottom'], ['awake', 'asleep']...].",
            },
            {"role": "assistant", "content": "Sure, here is a list of 100 antonyms: "},
        ],
    )
    return response


if os.environ.get("OPENAI_API_KEY", None) is not None:
    ANTONYM_PAIRS = generate_antonym_dataset(100)
    # Save the word pairs in a text file
    with open(section_dir / "data" / "my_antonym_pairs.txt", "w") as f:
        for word_pair in ANTONYM_PAIRS:
            f.write(f"{word_pair[0]} {word_pair[1]}\n")

# Load the word pairs from the text file
with open(section_dir / "data" / "antonym_pairs.txt", "r") as f:
    ANTONYM_PAIRS = [line.split() for line in f.readlines()]

print(ANTONYM_PAIRS[:10])

[['old', 'young'], ['top', 'bottom'], ['awake', 'asleep'], ['future', 'past'], ['appear', 'disappear'], ['early', 'late'], ['empty', 'full'], ['innocent', 'guilty'], ['ancient', 'modern'], ['arrive', 'depart']]


In [19]:
class ICLSequence:
    """
    Class to store a single antonym sequence.

    Uses the default template "Q: {x}\nA: {y}" (with separate pairs split by "\n\n").
    """

    def __init__(self, word_pairs: list[tuple[str, str]]):
        self.word_pairs = word_pairs
        self.x, self.y = zip(*word_pairs)

    def __len__(self):
        return len(self.word_pairs)

    def __getitem__(self, idx: int):
        return self.word_pairs[idx]

    def prompt(self):
        """Returns the prompt, which contains all but the second element in the last word pair."""
        p = "\n\n".join([f"Q: {x}\nA: {y}" for x, y in self.word_pairs])
        return p[: -len(self.completion())]

    def completion(self):
        """Returns the second element in the last word pair (with padded space)."""
        return " " + self.y[-1]

    def __str__(self):
        """Prints a readable string representation of the prompt & completion (indep of template)."""
        return f"{', '.join([f'({x}, {y})' for x, y in self[:-1]])}, {self.x[-1]} ->".strip(", ")


word_list = [["hot", "cold"], ["yes", "no"], ["in", "out"], ["up", "down"]]
seq = ICLSequence(word_list)

print("Tuple-representation of the sequence:")
print(seq)
print("\nActual prompt, which will be fed into the model:")
print(seq.prompt())

Tuple-representation of the sequence:
(hot, cold), (yes, no), (in, out), up ->

Actual prompt, which will be fed into the model:
Q: hot
A: cold

Q: yes
A: no

Q: in
A: out

Q: up
A:


In [20]:
class ICLDataset:
    """
    Dataset to create antonym pair prompts, in ICL task format. We use random seeds for consistency
    between the corrupted and clean datasets.

    Inputs:
        word_pairs:
            list of ICL task, e.g. [["old", "young"], ["top", "bottom"], ...] for the antonym task
        size:
            number of prompts to generate
        n_prepended:
            number of antonym pairs before the single-word ICL task
        bidirectional:
            if True, then we also consider the reversed antonym pairs
        corrupted:
            if True, then the second word in each pair is replaced with a random word
        seed:
            random seed, for consistency & reproducibility
    """

    def __init__(
        self,
        word_pairs: list[list[str]],
        size: int,
        n_prepended: int,
        bidirectional: bool = True,
        seed: int = 0,
        corrupted: bool = False,
    ):
        assert n_prepended + 1 <= len(word_pairs), "Not enough antonym pairs in dataset to create prompt."

        self.word_pairs = word_pairs
        self.word_list = [word for word_pair in word_pairs for word in word_pair]
        self.size = size
        self.n_prepended = n_prepended
        self.bidirectional = bidirectional
        self.corrupted = corrupted
        self.seed = seed

        self.seqs = []
        self.prompts = []
        self.completions = []

        # Generate the dataset (by choosing random word pairs, and constructing ICLSequence objects)
        for n in range(size):
            np.random.seed(seed + n)
            random_pairs = np.random.choice(len(self.word_pairs), n_prepended + 1, replace=False)
            # Randomize the order of each word pair (x, y).
            # If not bidirectional, we always have x -> y not y -> x
            random_orders = np.random.choice([1, -1], n_prepended + 1)
            if not (bidirectional):
                random_orders[:] = 1
            word_pairs = [self.word_pairs[pair][::order] for pair, order in zip(random_pairs, random_orders)]
            # If corrupted, then replace y with a random word in all (x, y) pairs except the last one
            if corrupted:
                for i in range(len(word_pairs) - 1):
                    word_pairs[i][1] = np.random.choice(self.word_list)
            seq = ICLSequence(word_pairs)

            self.seqs.append(seq)
            self.prompts.append(seq.prompt())
            self.completions.append(seq.completion())

    def create_corrupted_dataset(self):
        """Creates a corrupted version of the dataset (with same random seed)."""
        return ICLDataset(
            self.word_pairs,
            self.size,
            self.n_prepended,
            self.bidirectional,
            corrupted=True,
            seed=self.seed,
        )

    def __len__(self):
        return self.size

    def __getitem__(self, idx: int):
        return self.seqs[idx]

In [21]:
dataset = ICLDataset(ANTONYM_PAIRS, size=10, n_prepended=2, corrupted=False)

table = Table("Prompt", "Correct completion")
for seq, completion in zip(dataset.seqs, dataset.completions):
    table.add_row(str(seq), repr(completion))

rprint(table)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Prompt                                               ┃ Correct completion ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ (right, left), (maximum, minimum), melt ->           │ ' freeze'          │
│ (minimum, maximum), (old, new), punishment ->        │ ' reward'          │
│ (arrogant, humble), (blunt, sharp), compulsory ->    │ ' voluntary'       │
│ (inside, outside), (freeze, melt), full ->           │ ' empty'           │
│ (reject, accept), (awake, asleep), dusk ->           │ ' dawn'            │
│ (invisible, visible), (punishment, reward), heavy -> │ ' light'           │
│ (victory, defeat), (forward, backward), young ->     │ ' old'             │
│ (up, down), (compulsory, voluntary), right ->        │ ' wrong'           │
│ (open, closed), (domestic, foreign), brave ->        │ ' cowardly'        │
│ (under, over), (past, future), increase ->           │ ' decrease'        │
└──────────────────────────────────────────────────────┴────────────────────┘

In [ ]:
dataset = ICLDataset(ANTONYM_PAIRS, size=10, n_prepended=2, corrupted=True)

table = Table("Prompt", "Correct completion")
for seq, completions in zip(dataset.seqs, dataset.completions):
    table.add_row(str(seq), repr(completions))

rprint(table)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Prompt                                            ┃ Correct completion ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ (right, private), (maximum, destroy), melt ->     │ ' freeze'          │
│ (minimum, increase), (old, sharp), punishment ->  │ ' reward'          │
│ (arrogant, humble), (blunt, deep), compulsory ->  │ ' voluntary'       │
│ (inside, voluntary), (freeze, exterior), full ->  │ ' empty'           │
│ (reject, profit), (awake, start), dusk ->         │ ' dawn'            │
│ (invisible, birth), (punishment, spend), heavy -> │ ' light'           │
│ (victory, rich), (forward, honest), young ->      │ ' old'             │
│ (up, lie), (compulsory, short), right ->          │ ' wrong'           │
│ (open, soft), (domestic, anxious), brave ->       │ ' cowardly'        │
│ (under, melt), (past, young), increase ->         │ ' decrease'        │
└───────────────────────────────────────────────────┴────────────────────┘

In [30]:
word_pairs = list(zip(string.ascii_lowercase, string.ascii_uppercase))
dataset = ICLDataset(word_pairs, size=5, n_prepended=5, bidirectional=False, seed=0)

In [35]:
def calculate_h(model: LanguageModel, dataset: ICLDataset, layer: int = -1) -> tuple[list[str], Tensor]:
    """
    Averages over the model's hidden representations on each of the prompts in `dataset` at layer
    `layer`, to produce a single vector `h`.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        dataset: ICLDataset
            the dataset whose prompts `dataset.prompts` you're extracting the activations from (at
            the last seq pos)
        layer: int
            the layer you're extracting activations from

    Returns:
        completions: list[str]
            list of the model's next-token predictions (i.e. the strings the model predicts to
            follow the last token)
        h: Tensor
            average hidden state tensor at final sequence position, of shape (d_model,)
    """
    with model.trace(dataset.prompts, remote=REMOTE):
        h = model.transformer.h[layer].output[0][:, -1].mean(dim=0).save()
        logits = model.lm_head.output[:, -1]
        next_tok_id = logits.argmax(dim=-1).save()

    completions = model.tokenizer.batch_decode(next_tok_id.unsqueeze(-1))
    return completions, h

tests.test_calculate_h(calculate_h, model)

⬇ Downloading: 100%|██████████| 7.20k/7.20k [00:00<00:00]


All tests in `test_calculate_h` passed.


In [36]:
def display_model_completions_on_antonyms(
    model: LanguageModel,
    dataset: ICLDataset,
    completions: list[str],
    num_to_display: int = 20,
) -> None:
    table = Table(
        "Prompt (tuple representation)",
        "Model's completion\n(green=correct)",
        "Correct completion",
        title="Model's antonym completions",
    )

    for i in range(min(len(completions), num_to_display)):
        # Get model's completion, and correct completion
        completion = completions[i]
        correct_completion = dataset.completions[i]
        correct_completion_first_token = model.tokenizer.tokenize(correct_completion)[0].replace("Ġ", " ")
        seq = dataset.seqs[i]

        # Color code the completion based on whether it's correct
        is_correct = completion == correct_completion_first_token
        completion = f"[b green]{repr(completion)}[/]" if is_correct else repr(completion)

        table.add_row(str(seq), completion, repr(correct_completion))

    rprint(table)


# Get uncorrupted dataset
dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=2)

# Getting it from layer 12, as in the description in section 2.1 of paper
model_completions, h = calculate_h(model, dataset, layer=12)

# Displaying the output
display_model_completions_on_antonyms(model, dataset, model_completions)

⬇ Downloading: 100%|██████████| 7.24k/7.24k [00:00<00:00]


                                    Model's antonym completions                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃                                                       ┃ Model's completion ┃                    ┃
┃ Prompt (tuple representation)                         ┃ (green=correct)    ┃ Correct completion ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ (right, left), (maximum, minimum), melt ->            │ ' cast'            │ ' freeze'          │
│ (minimum, maximum), (old, new), punishment ->         │ ' reward'          │ ' reward'          │
│ (arrogant, humble), (blunt, sharp), compulsory ->     │ ' optional'        │ ' voluntary'       │
│ (inside, outside), (freeze, melt), full ->            │ ' empty'           │ ' empty'           │
│ (reject, accept), (awake, asleep), dusk ->            │ ' dawn'            │ ' dawn'            │
│ (invisible, visible), (punishment, reward), heavy ->  │ ' light'           │ ' light'           │
│ (victory, defeat), (forward, backward), young ->      │ ' old'             │ ' old'             │
│ (up, down), (compulsory, voluntary), right ->         │ ' wrong'           │ ' wrong'           │
│ (open, closed), (domestic, foreign), brave ->         │ ' cowardly'        │ ' cowardly'        │
│ (under, over), (past, future), increase ->            │ ' decrease'        │ ' decrease'        │
│ (inside, outside), (melt, freeze), over ->            │ ' under'           │ ' under'           │
│ (solid, liquid), (backward, forward), open ->         │ ' closed'          │ ' closed'          │
│ (optimist, pessimist), (invisible, visible), brave -> │ ' cowardly'        │ ' cowardly'        │
│ (noisy, quiet), (sell, buy), north ->                 │ ' south'           │ ' south'           │
│ (guilty, innocent), (birth, death), victory ->        │ ' defeat'          │ ' defeat'          │
│ (answer, question), (noisy, quiet), ancient ->        │ ' modern'          │ ' modern'          │
│ (on, off), (success, failure), flexible ->            │ ' rigid'           │ ' rigid'           │
│ (junior, senior), (arrive, depart), punishment ->     │ ' reward'          │ ' reward'          │
│ (loose, tight), (learn, teach), new ->                │ ' new'             │ ' old'             │
│ (introduce, remove), (deficiency, quality), wet ->    │ ' wet'             │ ' dry'             │
└───────────────────────────────────────────────────────┴────────────────────┴────────────────────┘

In [56]:
def intervene_with_h(
    model: LanguageModel,
    zero_shot_dataset: ICLDataset,
    h: Tensor,
    layer: int,
    remote: bool = REMOTE,
) -> tuple[list[str], list[str]]:
    """
    Extracts the vector `h` using previously defined function, and intervenes by adding `h` to the
    residual stream of a set of generated zero-shot prompts.

    Inputs:
        model: the model we're using to generate completions
        zero_shot_dataset: the dataset of zero-shot prompts which we'll intervene on, using the
            `h`-vector
        h: the `h`-vector we'll be adding to the residual stream
        layer: the layer we'll be extracting the `h`-vector from
        remote: whether to run the forward pass on the remote server (used for running test code)

    Returns:
        completions_zero_shot: list of string completions for the zero-shot prompts, without
            intervention using the h-vector
        completions_intervention: list of string completions for the zero-shot prompts, with
            intervention using the h-vector
    """
    
    with model.trace(remote=REMOTE) as tracer:
        with tracer.invoke(zero_shot_dataset.prompts):
            token_completions_zero_shot = model.lm_head.output[:, -1].argmax(dim=-1).save()
            
        with tracer.invoke(zero_shot_dataset.prompts):
            hidden_states = model.transformer.h[layer].output[0]
            hidden_states[:, -1] += h.to(hidden_states.device)
            token_completions_intervention = model.lm_head.output[:, -1].argmax(dim=-1).save()
            
    completions_zero_shot = model.tokenizer.batch_decode(token_completions_zero_shot)
    completions_interventions = model.tokenizer.batch_decode(token_completions_intervention)
    return completions_zero_shot, completions_interventions

In [ ]:
layer = 12
dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=3, seed=0)
zero_shot_dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=0, seed=1)

# Run previous function to get h-vector
h = calculate_h(model, dataset, layer=layer)[1]

# Run new function to intervene with h-vector
completions_zero_shot, completions_intervention = intervene_with_h(model, zero_shot_dataset, h, layer=layer)

print("Zero-shot completions: ", completions_zero_shot)
print("Completions with intervention: ", completions_intervention)

⬇ Downloading: 100%|██████████| 7.23k/7.23k [00:00<00:00]


⬇ Downloading: 100%|██████████| 729/729 [00:00<00:00]

Zero-shot completions:  [' minimum I inside reject invisible victory up open under inside solid\n noisy guilty yes I senior loose introduce innocent']
Completions with intervention:  [' maximum arrogant outside reject invisible victory down closed under outside solid optim noisy guilty answer on senior loose introduce guilty']


In [59]:
def display_model_completions_on_h_intervention(
    dataset: ICLDataset,
    completions: list[str],
    completions_intervention: list[str],
    num_to_display: int = 20,
) -> None:
    table = Table(
        "Prompt",
        "Model's completion\n(no intervention)",
        "Model's completion\n(intervention)",
        "Correct completion",
        title="Model's antonym completions",
    )

    for i in range(min(len(completions), num_to_display)):
        completion_ni = completions[i]
        completion_i = completions_intervention[i]
        correct_completion = dataset.completions[i]
        correct_completion_first_token = tokenizer.tokenize(correct_completion)[0].replace("Ġ", " ")
        seq = dataset.seqs[i]

        # Color code the completion based on whether it's correct
        is_correct = completion_i == correct_completion_first_token
        completion_i = f"[b green]{repr(completion_i)}[/]" if is_correct else repr(completion_i)

        table.add_row(str(seq), repr(completion_ni), completion_i, repr(correct_completion))

    rprint(table)


display_model_completions_on_h_intervention(zero_shot_dataset, completions_zero_shot, completions_intervention)

                                            Model's antonym completions                                            
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃            ┃ Model's completion                    ┃ Model's completion                    ┃                    ┃
┃ Prompt     ┃ (no intervention)                     ┃ (intervention)                        ┃ Correct completion ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ minimum -> │ ' minimum I inside reject invisible   │ ' maximum arrogant outside reject     │ ' maximum'         │
│            │ victory up open under inside solid\n  │ invisible victory down closed under   │                    │
│            │ noisy guilty yes I senior loose       │ outside solid optim noisy guilty      │                    │
│            │ introduce innocent'                   │ answer on senior loose introduce      │                    │
│            │                                       │ guilty'                               │                    │
└────────────┴───────────────────────────────────────┴───────────────────────────────────────┴────────────────────┘

In [70]:
def calculate_h_and_intervene_logprobs(
    model: LanguageModel,
    dataset: ICLDataset,
    zero_shot_dataset: ICLDataset,
    layer: int,
) -> tuple[list[str], list[str]]:
    """
    Extracts the vector `h`, intervenes by adding `h` to the residual stream of a set of generated
    zero-shot prompts, all within the same forward pass. Returns the completions from this
    intervention.

    Inputs:
        model: LanguageModel
            the model we're using to generate completions
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the `h`-vector
        zero_shot_dataset: ICLDataset
            the dataset of zero-shot prompts which we'll intervene on, using the `h`-vector
        layer: int
            the layer we'll be extracting the `h`-vector from

    Returns:
        completions_zero_shot: list[str]
            list of string completions for the zero-shot prompts, without intervention
        completions_intervention: list[str]
            list of string completions for the zero-shot prompts, with h-intervention
    """
    correct_completion_ids = [toks[0] for toks in tokenizer(zero_shot_dataset.completions)["input_ids"]]
    
    with model.trace(remote=REMOTE) as tracer:
        with tracer.invoke(dataset.prompts):
            h = model.transformer.h[layer].output[0][:, -1].mean(dim=0)
        
        with tracer.invoke(zero_shot_dataset.prompts):
            clean_logprobs = model.lm_head.output.log_softmax(dim=-1)[
                range(len(zero_shot_dataset)), -1, correct_completion_ids
            ].save()
        
        with tracer.invoke(zero_shot_dataset.prompts):
            hidden = model.transformer.h[layer].output[0]
            hidden[:, -1] += h
            intervene_logprobs = model.lm_head.output.log_softmax(dim=-1)[
                range(len(zero_shot_dataset)), -1, correct_completion_ids
            ].save()
            
    return clean_logprobs, intervene_logprobs


dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=3, seed=0)
zero_shot_dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=0, seed=1)

completions_zero_shot, completions_intervention = calculate_h_and_intervene_logprobs(
    model, dataset, zero_shot_dataset, layer=layer
)

display_model_completions_on_h_intervention(zero_shot_dataset, completions_zero_shot, completions_intervention)

⬇ Downloading: 100%|██████████| 700/700 [00:00<00:00]


                                            Model's antonym completions                                            
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃              ┃ Model's completion                   ┃ Model's completion                   ┃                    ┃
┃ Prompt       ┃ (no intervention)                    ┃ (intervention)                       ┃ Correct completion ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ minimum ->   │ tensor(-2.7500,                      │ tensor(-0.6367,                      │ ' maximum'         │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ arrogant ->  │ tensor(-6.1875,                      │ tensor(-3.9219,                      │ ' humble'          │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ inside ->    │ tensor(-3.6875,                      │ tensor(-0.9922,                      │ ' outside'         │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ reject ->    │ tensor(-3.9062,                      │ tensor(-1.9766,                      │ ' accept'          │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ invisible -> │ tensor(-3.7500,                      │ tensor(-1.9922,                      │ ' visible'         │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ victory ->   │ tensor(-4.4688,                      │ tensor(-2.2969,                      │ ' defeat'          │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ up ->        │ tensor(-3.9062,                      │ tensor(-1.2109,                      │ ' down'            │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ open ->      │ tensor(-5.0625,                      │ tensor(-1.4062,                      │ ' closed'          │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ under ->     │ tensor(-4.7188,                      │ tensor(-3.3750,                      │ ' over'            │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ inside ->    │ tensor(-3.6875,                      │ tensor(-0.9922,                      │ ' outside'         │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ solid ->     │ tensor(-5.5625,                      │ tensor(-3.0312,                      │ ' liquid'          │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ optimist ->  │ tensor(-6.4375,                      │ tensor(-3.3281,                      │ ' pessimist'       │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ noisy ->     │ tensor(-4.2188,                      │ tensor(-3.3438,                      │ ' quiet'           │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ guilty ->    │ tensor(-4.9375,                      │ tensor(-2.6875,                      │ ' innocent'        │
│              │ dtype=torch.bfloat16)                │ dtype=torch.bfloat16)                │                    │
│ answer ->    │ tensor(-5.1250,                      │ tensor(-3.9531,                      │ ' question'        │
│              │ dtype=torch.bfloat16)                │ 

In [71]:
def display_model_logprobs_on_h_intervention(
    dataset: ICLDataset,
    correct_logprobs_zero_shot: list[float],
    correct_logprobs_intervention: list[float],
    num_to_display: int = 20,
) -> None:
    table = Table(
        "Zero-shot prompt",
        "Model's logprob\n(no intervention)",
        "Model's logprob\n(intervention)",
        "Change in logprob",
        title="Model's antonym logprobs, with zero-shot h-intervention\n(green = intervention improves accuracy)",
    )

    for i in range(min(len(correct_logprobs_zero_shot), num_to_display)):
        logprob_ni = correct_logprobs_zero_shot[i]
        logprob_i = correct_logprobs_intervention[i]
        delta_logprob = logprob_i - logprob_ni
        zero_shot_prompt = f"{dataset[i].x[0]:>8} -> {dataset[i].y[0]}"

        # Color code the logprob based on whether it's increased with this intervention
        is_improvement = delta_logprob >= 0
        delta_logprob = f"[b green]{delta_logprob:+.2f}[/]" if is_improvement else f"{delta_logprob:+.2f}"

        table.add_row(zero_shot_prompt, f"{logprob_ni:.2f}", f"{logprob_i:.2f}", delta_logprob)

    rprint(table)


dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=3, seed=0)
zero_shot_dataset = ICLDataset(ANTONYM_PAIRS, size=20, n_prepended=0, seed=1)

correct_logprobs_zero_shot, correct_logprobs_intervention = calculate_h_and_intervene_logprobs(
    model, dataset, zero_shot_dataset, layer=layer
)

display_model_logprobs_on_h_intervention(
    zero_shot_dataset, correct_logprobs_zero_shot, correct_logprobs_intervention
)

⬇ Downloading: 100%|██████████| 700/700 [00:00<00:00]


              Model's antonym logprobs, with zero-shot h-intervention              
                     (green = intervention improves accuracy)                      
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃                       ┃ Model's logprob   ┃ Model's logprob ┃                   ┃
┃ Zero-shot prompt      ┃ (no intervention) ┃ (intervention)  ┃ Change in logprob ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│  minimum -> maximum   │ -2.75             │ -0.64           │ +2.11             │
│ arrogant -> humble    │ -6.19             │ -3.92           │ +2.27             │
│   inside -> outside   │ -3.69             │ -0.99           │ +2.69             │
│   reject -> accept    │ -3.91             │ -1.98           │ +1.93             │
│ invisible -> visible  │ -3.75             │ -1.99           │ +1.76             │
│  victory -> defeat    │ -4.47             │ -2.30           │ +2.17             │
│       up -> down      │ -3.91             │ -1.21           │ +2.69             │
│     open -> closed    │ -5.06             │ -1.41           │ +3.66             │
│    under -> over      │ -4.72             │ -3.38           │ +1.34             │
│   inside -> outside   │ -3.69             │ -0.99           │ +2.69             │
│    solid -> liquid    │ -5.56             │ -3.03           │ +2.53             │
│ optimist -> pessimist │ -6.44             │ -3.33           │ +3.11             │
│    noisy -> quiet     │ -4.22             │ -3.34           │ +0.88             │
│   guilty -> innocent  │ -4.94             │ -2.69           │ +2.25             │
│   answer -> question  │ -5.12             │ -3.95           │ +1.17             │
│       on -> off       │ -7.06             │ -4.38           │ +2.69             │
│   junior -> senior    │ -2.25             │ -1.07           │ +1.18             │
│    loose -> tight     │ -5.53             │ -2.94           │ +2.59             │
│ introduce -> remove   │ -7.47             │ -6.16           │ +1.31             │
│ innocent -> guilty    │ -2.84             │ -1.67           │ +1.17             │
└───────────────────────┴───────────────────┴─────────────────┴───────────────────┘

In [72]:
def calculate_fn_vectors_and_intervene(
    model: LanguageModel,
    dataset: ICLDataset,
    layers: list[int] | None = None,
) -> Float[Tensor, "layers heads"]:
    """
    Returns a tensor of shape (layers, heads), containing the CIE for each head.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the function vector (we'll also
            create a corrupted version of this dataset for interventions)
        layers: list[int] | None
            the layers which this function will calculate score for (if None, this means all layers)
    """
    layers = range(model.config.n_layer) if (layers is None) else layers
    heads = range(model.config.n_head)

    # Get corrupted dataset
    corrupted_dataset = dataset.create_corrupted_dataset()
    N = len(dataset)

    # Get correct token ids, so we can get correct token logprobs
    correct_completion_ids = [toks[0] for toks in tokenizer(dataset.completions)["input_ids"]]

    with model.trace(remote=REMOTE) as tracer:
        # Run a forward pass on clean prompts, where we store attention head outputs
        z_dict = {}
        with tracer.invoke(dataset.prompts):
            for layer in layers:
                # Get hidden states, reshape to get head dimension, store the mean tensor
                z = model.transformer.h[layer].attn.out_proj.input[:, -1]
                z_reshaped = z.reshape(N, N_HEADS, D_HEAD).mean(dim=0)
                for head in heads:
                    z_dict[(layer, head)] = z_reshaped[head]

        # Run a forward pass on corrupted prompts, where we don't intervene or store activations (just so we can get the
        # correct-token logprobs to compare with our intervention)
        with tracer.invoke(corrupted_dataset.prompts):
            logits = model.lm_head.output[:, -1]
            correct_logprobs_corrupted = logits.log_softmax(dim=-1)[t.arange(N), correct_completion_ids].save()

        # For each head, run a forward pass on corrupted prompts (here we need multiple different forward passes, since
        # we're doing different interventions each time)
        correct_logprobs_dict = {}
        for layer in layers:
            for head in heads:
                with tracer.invoke(corrupted_dataset.prompts):
                    # Get hidden states, reshape to get head dimension, then set it to the a-vector
                    z = model.transformer.h[layer].attn.out_proj.input[:, -1]
                    z.reshape(N, N_HEADS, D_HEAD)[:, head] = z_dict[(layer, head)]
                    # Get logprobs at the end, which we'll compare with our corrupted logprobs
                    logits = model.lm_head.output[:, -1]
                    correct_logprobs_dict[(layer, head)] = logits.log_softmax(dim=-1)[
                        t.arange(N), correct_completion_ids
                    ].save()

    # Get difference between intervention logprobs and corrupted logprobs, and take mean over batch dim
    all_correct_logprobs_intervention = einops.rearrange(
        t.stack([v for v in correct_logprobs_dict.values()]),
        "(layers heads) batch -> layers heads batch",
        layers=len(layers),
    )
    logprobs_diff = all_correct_logprobs_intervention - correct_logprobs_corrupted  # shape [layers heads batch]

    # Return mean effect of intervention, over the batch dimension
    return logprobs_diff.mean(dim=-1)

In [ ]:
dataset = ICLDataset(ANTONYM_PAIRS, size=8, n_prepended=2)


def batch_process_layers(n_layers, batch_size):
    for i in range(0, n_layers, batch_size):
        yield range(n_layers)[i : i + batch_size]


results = t.empty((0, N_HEADS), device=device)

# If this fails to run, you should reduce the batch size so the forward passes are split up more, or
# reduce dataset size
for layers in batch_process_layers(N_LAYERS, batch_size=4):
    print(f"Computing layers in {layers} ...")
    t0 = time.time()
    results = t.concat([results, calculate_fn_vectors_and_intervene(model, dataset, layers).to(device)])
    print(f"... finished in {time.time() - t0:.2f} seconds.\n")

imshow(
    results.T,
    title="Average indirect effect of function-vector intervention on antonym task",
    width=1000,
    height=600,
    labels={"x": "Layer", "y": "Head"},
    aspect="equal",
)

In [90]:
def calculate_fn_vector(
    model: LanguageModel,
    dataset: ICLDataset,
    head_list: list[tuple[int, int]],
) -> Float[Tensor, " d_model"]:
    """
    Returns a vector of length `d_model`, containing the sum of vectors written to the residual
    stream by the attention heads in `head_list`, averaged over all inputs in `dataset`.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        dataset: ICLDataset
            the dataset of clean prompts from which we'll extract the function vector (we'll also
            create a corrupted version of this dataset for interventions)
        head_list: list[tuple[int, int]]
            list of attention heads we're calculating the function vector from
    """
    # Turn head_list into a dict of {layer: heads we need in this layer}
    head_dict = defaultdict(set)
    for layer, head in head_list:
        head_dict[layer].add(head)

    fn_vector_list = []

    with model.trace(dataset.prompts, remote=REMOTE):
        for layer, head_list in head_dict.items():
            # Get the output projection layer
            out_proj = model.transformer.h[layer].attn.out_proj

            # Get the mean output projection input (note, setting values of this tensor will not
            # have downstream effects on other tensors)
            hidden_states = out_proj.input[:, -1].mean(dim=0)

            # Zero-ablate all heads which aren't in our list, then get the output (which
            # will be the sum over the heads we actually do want!)
            heads_to_ablate = set(range(N_HEADS)) - head_dict[layer]
            for head in heads_to_ablate:
                hidden_states.reshape(N_HEADS, D_HEAD)[head] = 0.0

            # Now that we've zeroed all unimportant heads, get the output & add it to the list
            # (we need a single batch dimension so we can use `out_proj`)
            out_proj_output = out_proj(hidden_states.unsqueeze(0)).squeeze()
            fn_vector_list.append(out_proj_output.save())

    # We sum all attention head outputs to get our function vector
    if not fn_vector_list:
        # Assuming D_MODEL is accessible in your scope
        return t.zeros(D_MODEL)
    
    fn_vector = sum([v for v in fn_vector_list])

    assert fn_vector.shape == (D_MODEL,)
    return fn_vector

In [95]:
def intervene_with_fn_vector(
    model: LanguageModel,
    word: str,
    layer: int,
    fn_vector: Float[Tensor, " d_model"],
    prompt_template='The word "{x}" means',
    n_tokens: int = 5,
) -> tuple[str, str]:
    """
    Intervenes with a function vector, by adding it at the last sequence position of a generated
    prompt.

    Inputs:
        model: LanguageModel
            the transformer you're doing this computation with
        word: str
            The word substituted into the prompt template, via prompt_template.format(x=word)
        layer: int
            The layer at which we'll make the intervention (by adding the function vector)
        fn_vector: Float[Tensor, "d_model"]
            The vector we'll add to the final sequence position for each new token to be generated
        prompt_template:
            The template of the prompt we'll use to produce completions
        n_tokens: int
            The number of additional tokens we'll generate for our unsteered / steered completions

    Returns:
        completion: str
            The full completion (including original prompt) for the no-intervention case
        completion_intervention: str
            The full completion (including original prompt) for the intervention case
    """
    
    prompt = prompt_template.format(x=word)
    with model.generate(remote=REMOTE, max_new_tokens=n_tokens, repetition_penalty=1.2) as generator:
        with model.trace(dataset.prompts, remote=REMOTE):
            with generator.invoke(prompt):
                tokens = model.generator.output.save()
                
            with generator.invoke(prompt):
                model.transformer.h[layer].output[0][:, -1] += fn_vector
                tokens_intervention = model.generator.output.save()

    completion, completion_intervention = tokenizer.batch_decode(
        [tokens.squeeze().tolist(), tokens_intervention.squeeze().tolist()]
    )                
    
    return completion, completion_intervention 
            

In [ ]:
# Remove word from our pairs, so it can be a holdout
word = "light"
_ANTONYM_PAIRS = [pair for pair in ANTONYM_PAIRS if word not in pair]

# Define our dataset, and the attention heads we'll use
dataset = ICLDataset(_ANTONYM_PAIRS, size=20, n_prepended=5)
head_list = [
    (8, 0),
    (8, 1),
    (9, 14),
    (11, 0),
    (12, 10),
    (13, 12),
    (13, 13),
    (14, 9),
    (15, 5),
    (16, 14),
]

# Extract the function vector
fn_vector = calculate_fn_vector(model, dataset, head_list)

# Intervene with the function vector
completion, completion_intervention = intervene_with_fn_vector(
    model,
    word=word,
    layer=9,
    fn_vector=1.5 * fn_vector,
    prompt_template='The word "{x}" means',
    n_tokens=40,
)

table = Table("No intervention", "intervention")
table.add_row(repr(completion), repr(completion_intervention))
rprint(table)

In [ ]:
with open(section_dir / "data/country_capital_pairs.txt", "r", encoding="utf-8") as f:
    COUNTRY_CAPITAL_PAIRS = [line.split() for line in f.readlines()]

country = "Netherlands"
_COUNTRY_CAPITAL_PAIRS = [pair for pair in COUNTRY_CAPITAL_PAIRS if pair[0] != country]

dataset = ICLDataset(_COUNTRY_CAPITAL_PAIRS, size=20, n_prepended=5, bidirectional=False)
head_list = [
    (8, 0),
    (8, 1),
    (9, 14),
    (11, 0),
    (12, 10),
    (13, 12),
    (13, 13),
    (14, 9),
    (15, 5),
    (16, 14),
]

fn_vector = calculate_fn_vector(model, dataset, head_list)

# Intervene with the function vector
completion, completion_intervention = intervene_with_fn_vector(
    model=model,
    word=country,
    layer=9,
    fn_vector=fn_vector,
    prompt_template="When you think of {x},",
    n_tokens=40,
)

table = Table("No intervention", "intervention")
table.add_row(repr(completion), repr(completion_intervention))
rprint(table)

In [2]:
gpt2_xl = LanguageModel("gpt2-xl", device_map="auto", torch_dtype=t.bfloat16)
tokenizer = gpt2_xl.tokenizer

REMOTE = False
# If you are using gpt2_xl, set REMOTE = False as gpt2_xl is not hosted remotely by nnsight. You can
# set REMOTE = True for a remotely hosted model here (https://nnsight.net/status/)

In [7]:
SAMPLING_KWARGS = {
    "do_sample": True,
    "top_p": 0.3,
    "repetition_penalty": 1.2,
}


def calculate_and_apply_steering_vector(
    model: LanguageModel,
    prompt: str,
    activation_additions: list[tuple[int, float, str]],
    n_tokens: int,
    n_comparisons: int = 1,
    use_bos: bool = True,
) -> tuple[list[str], list[str]]:
    """
    Performs the steering vector experiments described in the LessWrong post.

    Args:
        model: LanguageModel
            the transformer you're doing this computation with
        prompt: str
            The original prompt, which we'll be doing activation steering on.

        activation_additions: list[tuple[int, float, str]], each tuple contains:
            layer - the layer we're applying these steering vectors to
            coefficient - the value we're multiplying it by
            prompt - the prompt we're inputting
            e.g. activation_additions[0] = [6, 5.0, " Love"] means we add the " Love" vector at
            layer 6, scaled by 5x

        n_tokens: int
            Number of tokens which will be generated for each completion

        n_comparisons: int
            Number of sequences generated in this function (i.e. we generate `n_comparisons` which
            are unsteered, and the same number which are steered).

    Returns:
        unsteered_completions: list[str]
            List of length `n_comparisons`, containing all the unsteered completions.

        steered_completions: list[str]
            List of length `n_comparisons`, containing all the steered completions.
    """
    # Add the BOS token manually, if we're including it
    if use_bos:
        bos = model.tokenizer.bos_token
        prompt = bos + prompt
        activation_additions = [[layer, coeff, bos + p] for layer, coeff, p in activation_additions]

    # Get the (layers, coeffs, prompts) in an easier form to use, also calculate the prompt lengths
    # and check they're all the same
    act_add_layers, act_add_coeffs, act_add_prompts = zip(*activation_additions)
    act_add_seq_lens = [len(tokenizer.tokenize(p)) for p in act_add_prompts]
    assert len(set(act_add_seq_lens)) == 1, "All activation addition prompts must be the same length."
    assert act_add_seq_lens[0] <= len(tokenizer.tokenize(prompt)), (
        "All act_add prompts should be shorter than original prompt."
    )

    # Get the prompts we'll intervene on (unsteered and steered)
    steered_prompts = [prompt for _ in range(n_comparisons)]
    unsteered_prompts = [prompt for _ in range(n_comparisons)]

    with model.generate(max_new_tokens=n_tokens, remote=REMOTE, **SAMPLING_KWARGS) as generator:
        # Run the act_add prompts (i.e. the contrast pairs), and extract their activations
        with generator.invoke(act_add_prompts):
            # Get all the prompts from the activation additions, and put them in a list
            # (note, we slice from the end of the sequence because of left-padding)
            act_add_vectors = [
                model.transformer.h[layer].output[0][i, -seq_len:]
                for i, (layer, seq_len) in enumerate(zip(act_add_layers, act_add_seq_lens))
            ]

        # Forward pass on unsteered prompts (no intervention, no activations saved - we only need
        # the completions)
        with generator.invoke(unsteered_prompts):
            unsteered_out = model.generator.output.save()

        # Forward pass on steered prompts (we add in the results from the act_add prompts)
        with generator.invoke(steered_prompts):
            # For each act_add prompt, add the vector to residual stream, at the start of the seq
            for i, (layer, coeff, seq_len) in enumerate(zip(act_add_layers, act_add_coeffs, act_add_seq_lens)):
                model.transformer.h[layer].output[0][:, :seq_len] += coeff * act_add_vectors[i]
            steered_out = model.generator.output.save()

    # Decode steered & unsteered completions (discarding the sequences we only used for extracting
    # activations) & return results
    unsteered_completions = tokenizer.batch_decode(unsteered_out[-n_comparisons:])
    steered_completions = tokenizer.batch_decode(steered_out[-n_comparisons:])

    return unsteered_completions, steered_completions

In [4]:
unsteered_completions, steered_completions = calculate_and_apply_steering_vector(
    gpt2_xl,
    prompt="I hate you because",
    activation_additions=[(6, +8.0, "Love "), (6, -8.0, "Hate")],
    n_tokens=50,
    n_comparisons=3,
    use_bos=True,
)

table = Table("Unsteered", "Steered", title="Completions", show_lines=True)
for usc, sc in zip(unsteered_completions, steered_completions):
    table.add_row(usc, sc)
rprint(table)

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

: 

In [8]:
unsteered_completions, steered_completions = calculate_and_apply_steering_vector(
    gpt2_xl,
    prompt="I went up to my friend and said",
    activation_additions=[
        (20, +4.0, "I talk about weddings constantly  "),
        (20, -4.0, "I do not talk about weddings constantly"),
    ],
    n_tokens=50,
    n_comparisons=3,
    use_bos=False,
)

table = Table("Unsteered", "Steered", title="Completions", show_lines=True)
for usc, sc in zip(unsteered_completions, steered_completions):
    table.add_row(usc, sc)
rprint(table)

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

: 

In [4]:
# Code to calculate decoded vocabulary:
logits = gpt2_xl._model.lm_head(fn_vector)
max_logits = logits.topk(20).indices.tolist()
tokens = gpt2_xl.tokenizer.batch_decode(max_logits)
print("Top logits:\n" + "\n".join(map(repr, tokens)))

NameError: name 'fn_vector' is not defined